# PHATE Visualization — 120k Article Embeddings
**Pipeline:** `train.csv` → clean titles → embed → PCA → PHATE → HDBSCAN → **offline precompute** → hybrid search

**Offline precompute** (Cell 9) stores:
- `cluster_centroids` — L2-normalised mean embedding per cluster
- `cluster_to_article_ids` — inverted index: cluster label → article indices
- `article_top_cluster_pos` — each article's top-k nearest cluster positions

**Online query** (Cell 13):
1. Embed query → cosine sim vs centroids → select top `TOP_CLUSTERS` positions
2. Candidate pool = hard-cluster members **∪** overlap members (articles whose precomputed top-k includes a selected cluster) **∪** noise
3. Cosine rerank all candidates → top-K results

Run cells top-to-bottom on first use. On re-runs you can skip straight to **Cell 4** if `embeddings_cache.npz` is already saved to Drive.


## Cell 1 — Install dependencies

In [1]:
!pip install phate
!pip install scanpy

## Cell 2 — Mount Google Drive & set paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ── Edit these two lines to match your Drive layout ──────────────────────────
CSV_PATH        = Path('/content/train.csv')
DRIVE_OUTPUT    = Path('/content/drive/MyDrive/phate_output')   # results saved here
# ─────────────────────────────────────────────────────────────────────────────

DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE_PATH            = DRIVE_OUTPUT / 'embeddings_cache.npz'      # skip embedding on re-runs

assert CSV_PATH.exists(), f'train.csv not found at {CSV_PATH}'
print(f'CSV      : {CSV_PATH}')
print(f'Output   : {DRIVE_OUTPUT}')
print(f'Cache    : {CACHE_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CSV      : /content/train.csv
Output   : /content/drive/MyDrive/phate_output
Cache    : /content/drive/MyDrive/phate_output/embeddings_cache.npz


## Cell 3 — Imports & config

In [3]:
import csv, html, re, time
from typing import Iterable

import numpy as np
import pandas as pd
import phate
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
import hdbscan
import torch

# ── Config — tweak here ───────────────────────────────────────────────────────
MODEL      = 'BAAI/bge-large-en-v1.5'   # → 1024-dim embeddings
BATCH_SIZE = 128                         # 64 for T4, 256 for A100
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

PCA_COMPONENTS           = 405    # dims to reduce to before PHATE
PHATE_KNN                = 15     # neighbours for affinity graph
PHATE_DECAY              = 40     # alpha-decay sharpness
PHATE_T                  = 'auto' # diffusion time
PHATE_COMPONENTS         = 2      # 2 = flat plot, 3 = 3-D plot
HDBSCAN_MIN_CLUSTER_SIZE = 6
COLOR_COLUMN             = None   # set to a CSV column name, e.g. 'Category',
                                  # or None to color by cluster label
FORCE_RECOMPUTE          = False  # set True to ignore embedding cache
# ─────────────────────────────────────────────────────────────────────────────

print(f'Device : {DEVICE}')
print(f'Model  : {MODEL}')
print('Config OK.')

Device : cuda
Model  : BAAI/bge-large-en-v1.5
Config OK.


## Cell 4 — Text cleaning helpers

In [4]:
import csv, html, re, time
from typing import Iterable

import numpy as np
import pandas as pd
import phate
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
import hdbscan
import torch

# ── Config — tweak here ───────────────────────────────────────────────────────
MODEL      = 'BAAI/bge-large-en-v1.5'   # → 1024-dim embeddings
BATCH_SIZE = 64                         # 64 for T4, 256 for A100
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

PCA_COMPONENTS           = 405    # dims to reduce to before PHATE
PHATE_KNN                = 15     # neighbours for affinity graph
PHATE_DECAY              = 40     # alpha-decay sharpness
PHATE_T                  = 'auto' # diffusion time
PHATE_COMPONENTS         = 2      # 2 = flat plot, 3 = 3-D plot
HDBSCAN_MIN_CLUSTER_SIZE = 6
COLOR_COLUMN             = None   # set to a CSV column name, e.g. 'Category',
                                  # or None to color by cluster label
FORCE_RECOMPUTE          = False  # set True to ignore embedding cache
# ─────────────────────────────────────────────────────────────────────────────

print(f'Device : {DEVICE}')
print(f'Model  : {MODEL}')
print('Config OK.')

# ------------------------------------------------------------------
# Source definitions
# ------------------------------------------------------------------

_NEWS_SOURCES = [
    'Reuters',
    'AP',
    'Associated Press',
    'AFP',
    'CNN',
    'BBC',
    'FOX',
    'MSNBC',
    'ABC News',
    'CBS News',
    'NBC News',
    'The Guardian',
    'The Times',
    'Xinhua',
    'SPACE.com',
    'MacCentral',
    'TechWeb',
    'PC World',
    'Wired News',
]

# Escape everything automatically so dots etc.
# become regex-safe.
_SOURCE_PATTERN = '|'.join(
    re.escape(source)
    for source in sorted(_NEWS_SOURCES, key=len, reverse=True)
)

# ------------------------------------------------------------------
# Generic cleanup
# ------------------------------------------------------------------

_RE_HTML_TAG = re.compile(r'<[^>]+>')
_RE_SPACES = re.compile(r'\s+')

_RE_EMBED_PUNCT = re.compile(r'[^a-zA-Z0-9\s]')

_RE_EMPTY_PARENS = re.compile(r'\(\s*\)')
_RE_EMPTY_BRACKS = re.compile(r'\[\s*\]')


# ------------------------------------------------------------------
# Source cleanup
# ------------------------------------------------------------------

# Reuters - Some title
_RE_SOURCE_PREFIX = re.compile(
    rf'^(?:{_SOURCE_PATTERN})\s*[-–]\s*',
    re.IGNORECASE,
)

# (Reuters)
# (AFP)
# (Associated Press)
_RE_SOURCE_PARENS = re.compile(
    rf'\(\s*(?:{_SOURCE_PATTERN})\s*\)',
    re.IGNORECASE,
)

# Reuters
# AP
# AFP
_RE_WIRE_TAGS = re.compile(
    rf'\b(?:{_SOURCE_PATTERN})\b',
    re.IGNORECASE,
)

# Some title - cnn.com
_RE_SOURCE_SUFFIX = re.compile(
    r'\s+[-–]\s+\S+\.(?:com|org|net|co\.uk|co|io|gov|edu)\s*$',
    re.IGNORECASE,
)

# ------------------------------------------------------------------
# HTML decoding
# ------------------------------------------------------------------

def _decode_html(text: str) -> str:
    """
    Handles:
        &amp;
        &lt;
        &#39;
        &amp;lt;
        &amp;amp;
        #39; (added handling)
    """
    text = re.sub(r'#39;', "'", text) # Handle specific #39; pattern
    text = html.unescape(text)
    text = html.unescape(text)
    return text

# ------------------------------------------------------------------
# Human-readable title
# ------------------------------------------------------------------

def clean_title_nlp(raw: str) -> str:

    s = _decode_html(raw)
    # Remove HTML tags
    s = _RE_HTML_TAG.sub(' ', s)
    # Replace 'amp;' with '&'
    s = re.sub(r'amp;', '&', s, flags=re.IGNORECASE)
    # Remove Reuters - prefix
    s = _RE_SOURCE_PREFIX.sub('', s)
    # Remove trailing site name
    s = _RE_SOURCE_SUFFIX.sub('', s)
    # Remove standalone AP / Reuters mentions
    s = _RE_WIRE_TAGS.sub(' ', s)
    # Normalize spaces
    s = _RE_SPACES.sub(' ', s)
    s = _RE_EMPTY_PARENS.sub('', s)
    s = _RE_EMPTY_BRACKS.sub('', s)
    return s.strip()

# ------------------------------------------------------------------
# Embedding title
# ------------------------------------------------------------------

def clean_title_embed(raw: str) -> str:

    s = clean_title_nlp(raw)
    s = s.lower()
    s = _RE_EMBED_PUNCT.sub(' ', s)
    s = _RE_SPACES.sub(' ', s)
    return s.strip()


# ------------------------------------------------------------------
# CSV loader
# ------------------------------------------------------------------

def load_titles_from_csv(csv_file: Path) -> tuple[list[str], pd.DataFrame]:
    """
    Returns
    -------
    embed_titles : list[str]

        Aggressively cleaned titles for embeddings.

    meta : DataFrame

        Original metadata plus:

        title_embed
        title_nlp
        article_body
    """

    print(f'[0] Reading {csv_file} ...')

    t0 = time.time()

    embed_titles = []
    rows = []

    with csv_file.open(
        'r',
        encoding='utf-8-sig',
        errors='replace',
        newline=''
    ) as f:

        reader = csv.DictReader(f)

        all_columns = reader.fieldnames or []

        body_col = all_columns[-1] if all_columns else None

        if body_col:
            print(f'    Article body column detected: "{body_col}"')
        else:
            print('    Could not detect article body column.')

        for row in reader:

            raw_title = (row.get('Title') or '').strip()

            if not raw_title:
                continue

            embed_title = clean_title_embed(raw_title)

            if not embed_title:
                continue

            embed_titles.append(embed_title)
            rows.append(row)

    meta = pd.DataFrame(rows)

    meta['title_embed'] = meta['Title'].apply(clean_title_embed)
    meta['title_nlp'] = meta['Title'].apply(clean_title_nlp)

    if body_col and body_col in meta.columns:
        meta['article_body'] = (
            meta[body_col]
            .fillna('')
            .astype(str)
            .str.strip()
        )
    else:
        meta['article_body'] = ''

    print(
        f'    {len(embed_titles):,} titles loaded'
        f'  |  {time.time() - t0:.1f}s'
    )

    print(f'    Columns: {list(meta.columns)}')

    return embed_titles, meta


print('Cleaning helpers defined.')

Device : cuda
Model  : BAAI/bge-large-en-v1.5
Config OK.
Cleaning helpers defined.


## Cell 5 — Load & clean titles from CSV

In [5]:
embed_titles, meta = load_titles_from_csv(CSV_PATH)

# Sanity-check: show both versions side by side
print('\nSample titles (embed vs nlp):')
for e, n in zip(embed_titles[:5], meta['title_nlp'][:5]):
    print(f'  embed : {e}')
    print(f'  nlp   : {n}')
    print()

[0] Reading /content/train.csv ...
    Article body column detected: "Description"
    120,000 titles loaded  |  8.4s
    Columns: ['Class Index', 'Title', 'Description', 'title_embed', 'title_nlp', 'article_body']

Sample titles (embed vs nlp):
  embed : wall st bears claw back into the black
  nlp   : Wall St. Bears Claw Back Into the Black

  embed : carlyle looks toward commercial aerospace
  nlp   : Carlyle Looks Toward Commercial Aerospace

  embed : oil and economy cloud stocks outlook
  nlp   : Oil and Economy Cloud Stocks' Outlook

  embed : iraq halts oil exports from main southern pipeline
  nlp   : Iraq Halts Oil Exports from Main Southern Pipeline

  embed : oil prices soar to all time record posing new menace to us economy
  nlp   : Oil prices soar to all-time record, posing new menace to US economy



## Cell 6 — Embed titles  *(skipped automatically if cache exists)*

In [6]:
def embed_titles(
    titles: list[str],
    model_name: str,
    batch_size: int,
    device: str,
) -> np.ndarray:
    print(f'[1] Embedding {len(titles):,} titles with {model_name!r}')
    print(f'    batch_size={batch_size}  device={device}')
    t = time.time()
    model = SentenceTransformer(model_name, device=device)
    emb = model.encode(
        titles,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,   # L2-normalise so cosine sim = dot product
    ).astype(np.float32)
    print(f'    Shape: {emb.shape}  |  RAM: ~{emb.nbytes/1e9:.2f} GB  |  {(time.time()-t)/60:.1f} min')
    return emb


# Retrieve the list of titles from the global `meta` DataFrame
# The global variable `embed_titles` (the list) was shadowed by the function definition in this cell.
titles_for_embedding = meta['title_embed'].tolist()

if not FORCE_RECOMPUTE and CACHE_PATH.exists():
    print(f'[1] Loading cached embeddings from {CACHE_PATH}')
    embeddings = np.load(CACHE_PATH)['embeddings'].astype(np.float32)
    print(f'    Shape: {embeddings.shape}')
    if len(embeddings) != len(titles_for_embedding):
        print('    ⚠  Cache size mismatch — recomputing ...')
        embeddings = embed_titles(titles_for_embedding, MODEL, BATCH_SIZE, DEVICE)
        np.savez_compressed(CACHE_PATH, embeddings=embeddings)
else:
    embeddings = embed_titles(titles_for_embedding, MODEL, BATCH_SIZE, DEVICE)
    np.savez_compressed(CACHE_PATH, embeddings=embeddings)
    print(f'    Cached → {CACHE_PATH}')

[1] Loading cached embeddings from /content/drive/MyDrive/phate_output/embeddings_cache.npz
    Shape: (120000, 1024)


## Cell 7 — PCA pre-reduction

In [7]:
# embeddings are L2-normalised (see Cell 6) — PCA operates in cosine space
t0 = time.time()

pca = PCA(n_components=0.95, random_state=42)
reduced = pca.fit_transform(embeddings).astype(np.float32)
PCA_COMPONENTS = pca.n_components_
print(f"Components needed for 95% variance: {pca.n_components_}")

explained = np.cumsum(pca.explained_variance_ratio_)
print(f'    Variance explained: {explained[-1]*100:.1f}%  |  {time.time()-t0:.1f}s')

# # Scree plot
# fig, ax = plt.subplots(figsize=(8, 3))
# ax.plot(range(1, pca.n_components_ + 1), explained * 100, lw=1.5, color='#534AB7')
# ax.axhline(95, color='#D85A30', ls='--', lw=1, label='95% threshold')
# ax.set_xlabel('Principal components')
# ax.set_ylabel('Cumulative variance (%)')
# ax.set_title('PCA scree — pick n_components where curve flattens')
# ax.legend()
# plt.tight_layout()

Components needed for 95% variance: 406
    Variance explained: 95.0%  |  25.0s


## Cell 8 — Fit PHATE
> ⏱ **Runtime:** ~5–15 min on a T4 GPU for 120k × 100 dims.  
> Key params: `PHATE_KNN` (5–30), `PHATE_DECAY` (10–100), `PHATE_T` ('auto' or int).

In [8]:
print(f'[3] Fitting PHATE  (knn={PHATE_KNN}, decay={PHATE_DECAY}, '
      f't={PHATE_T}, n_components={PHATE_COMPONENTS})')
t0 = time.time()

phate_op = phate.PHATE(
    n_components=PHATE_COMPONENTS,
    knn=PHATE_KNN,
    decay=PHATE_DECAY,
    t=PHATE_T,
    n_jobs=-1,
    random_state=42,
    verbose=True,
)
phate_emb = phate_op.fit_transform(reduced)

print(f'    Done in {(time.time()-t0)/60:.1f} min')
print(f'    Embedding shape: {phate_emb.shape}')

[3] Fitting PHATE  (knn=15, decay=40, t=auto, n_components=2)
Calculating PHATE...
  Running PHATE on 120000 observations and 406 variables.
  Calculating graph and diffusion operator...
    Calculating PCA...
    Calculated PCA in 5.43 seconds.
    Calculating KNN search...
    Calculated KNN search in 156.99 seconds.
    Calculating affinities...


/usr/local/lib/python3.12/dist-packages/graphtools/graphs.py:810: RuntimeWarning: Detected zero distance between 1513 pairs of samples. Consider removing duplicates to avoid errors in downstream processing.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/graphtools/graphs.py:505: RuntimeWarning: divide by zero encountered in divide
  scaled = distances_i / bw
/usr/local/lib/python3.12/dist-packages/graphtools/graphs.py:505: RuntimeWarning: invalid value encountered in divide
  scaled = distances_i / bw
/usr/local/lib/python3.12/dist-packages/graphtools/graphs.py:540: RuntimeWarning: invalid value encountered in divide
  scaled = distances_i / bw


    Calculated affinities in 10.87 seconds.
  Calculated graph and diffusion operator in 174.61 seconds.
  Calculating landmark operator...
    Calculating SVD...
    Calculated SVD in 40.99 seconds.
    Calculating KMeans...
    Calculated KMeans in 16.91 seconds.
  Calculated landmark operator in 57.90 seconds.
  Calculating optimal t...
    Automatically selected t = 39
  Calculated optimal t in 4.17 seconds.
  Calculating diffusion potential...
  Calculated diffusion potential in 2.99 seconds.
  Calculating metric MDS...
    SGD-MDS may not have converged: stress changed by -5.4% in final iterations. Consider increasing n_iter or adjusting learning_rate.
  Calculated metric MDS in 5.28 seconds.
Calculated PHATE in 248.29 seconds.
    Done in 4.1 min
    Embedding shape: (120000, 2)


## Cell 9 — Cluster in PHATE space (HDBSCAN) + Offline Precompute
Offline step builds three data structures used at query time:
- `cluster_centroids` — L2-normalised mean embedding per cluster
- `cluster_to_article_ids` — dict mapping cluster label → array of article indices
- `article_top_cluster_pos` — for each article, the top-k nearest cluster positions (for overlap expansion)

These are also saved to Drive so the offline step can be skipped on re-runs.


In [9]:
print(f'[4] HDBSCAN  (min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE})')
t0 = time.time()

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=5,
    cluster_selection_epsilon=0.0001,
)
labels = clusterer.fit_predict(phate_emb)
meta['cluster'] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_pct  = (labels == -1).mean() * 100
print(f'    {n_clusters} clusters found  |  noise: {noise_pct:.1f}%  |  {time.time()-t0:.1f}s')

# ═══════════════════════════════════════════════════════════════════════════════
# OFFLINE PRECOMPUTE — run once, reused every query
# Stores:
#   cluster_centroids        (K, dim)  — L2-normalised mean per cluster
#   cluster_to_article_ids   dict[int → np.ndarray]  — cluster → article indices
#   article_top_cluster_pos  (N, TOP_K_PER_ARTICLE)  — per-article nearest cluster positions
# ═══════════════════════════════════════════════════════════════════════════════
from sklearn.metrics.pairwise import cosine_similarity

# ── 1. Cluster centroids ──────────────────────────────────────────────────────
unique_clusters   = sorted(c for c in set(labels) if c != -1)
cluster_id_map    = np.array(unique_clusters)          # position → real label

cluster_centroids = np.stack([
    embeddings[labels == c].mean(axis=0)
    for c in unique_clusters
]).astype(np.float32)                                  # (K, dim)

_cn = np.linalg.norm(cluster_centroids, axis=1, keepdims=True)
cluster_centroids /= np.where(_cn == 0, 1.0, _cn)     # L2-normalise in-place

print(f'    Centroids : {cluster_centroids.shape}')

# ── 2. cluster → article ids (inverted index) ─────────────────────────────────
cluster_to_article_ids = {
    c: np.where(labels == c)[0].astype(np.int32)
    for c in unique_clusters
}
print(f'    Inverted index : {len(cluster_to_article_ids)} clusters')

# ── 3. article → top-k nearest cluster positions (overlap memberships) ────────
# Each article stores the POSITIONS (not labels) of its top-k nearest centroids
# so that at query time we can look up candidate sets in O(N_candidates) time.
TOP_K_CLUSTERS_PER_ARTICLE = 3    # tune: 2–5 is usually enough

CHUNK = 5_000
article_top_cluster_pos = np.empty(
    (len(embeddings), TOP_K_CLUSTERS_PER_ARTICLE), dtype=np.int32
)

for start in range(0, len(embeddings), CHUNK):
    end   = min(start + CHUNK, len(embeddings))
    sims  = cosine_similarity(embeddings[start:end], cluster_centroids)  # (chunk, K)
    article_top_cluster_pos[start:end] = np.argsort(sims, axis=1)[:, ::-1][:, :TOP_K_CLUSTERS_PER_ARTICLE]

print(f'    article_top_cluster_pos : {article_top_cluster_pos.shape}  '
      f'(top-{TOP_K_CLUSTERS_PER_ARTICLE} cluster positions per article)')

# ── 4. Save precomputed structures to Drive ───────────────────────────────────
PRECOMPUTE_PATH = DRIVE_OUTPUT / 'search_index.npz'

np.savez_compressed(
    PRECOMPUTE_PATH,
    cluster_centroids       = cluster_centroids,
    cluster_id_map          = cluster_id_map,
    article_top_cluster_pos = article_top_cluster_pos,
    # Inverted index: flatten to two arrays (offsets + values) for npz
    inv_idx_labels  = np.array(list(cluster_to_article_ids.keys()),   dtype=np.int32),
    inv_idx_offsets = np.concatenate([[0], np.cumsum([len(v) for v in cluster_to_article_ids.values()])]).astype(np.int32),
    inv_idx_values  = np.concatenate(list(cluster_to_article_ids.values())).astype(np.int32),
)
print(f'    Precomputed index saved → {PRECOMPUTE_PATH}')


[4] HDBSCAN  (min_cluster_size=6)
    2220 clusters found  |  noise: 27.4%  |  7.6s
    Centroids : (2220, 1024)
    Inverted index : 2220 clusters
    article_top_cluster_pos : (120000, 3)  (top-3 cluster positions per article)
    Precomputed index saved → /content/drive/MyDrive/phate_output/search_index.npz


In [10]:
import torch

print(f"Current DEVICE setting: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available() and DEVICE == 'cuda':
    print("\nWarning: CUDA is not available, but DEVICE is set to 'cuda'.\n"+
          "This might indicate an issue with your runtime environment or GPU connection.")
    print("Please check your Colab runtime type (Runtime > Change runtime type > GPU).")
elif DEVICE == 'cpu':
    print("\nNote: DEVICE is explicitly set to 'cpu'. If you intend to use a GPU,\n"+
          "please change the DEVICE variable in Cell 3 or 4 to 'cuda' and ensure you are connected to a GPU runtime.")


Current DEVICE setting: cuda
CUDA available: True


## Cell 10 — Interactive 2-D Plotly scatter

In [ ]:
MAX_INTERACTIVE = 30_000   # Plotly handles ~30k points smoothly in a browser

color_vals  = meta[COLOR_COLUMN].values if (COLOR_COLUMN and COLOR_COLUMN in meta.columns) else labels
color_label = COLOR_COLUMN or 'cluster'
hover_text  = meta['title_nlp'].tolist()

n = len(phate_emb)
if n > MAX_INTERACTIVE:
    print(f'Subsampling {n:,} → {MAX_INTERACTIVE:,} for interactive plot (stratified by cluster)')
    idx = []
    for c in np.unique(labels):
        ci = np.where(labels == c)[0]
        idx.append(np.random.choice(ci, min(MAX_INTERACTIVE // max(1, len(np.unique(labels))), len(ci)), replace=False))
    idx = np.concatenate(idx)[:MAX_INTERACTIVE]
    p_sub = phate_emb[idx];  c_sub = color_vals[idx];  h_sub = [hover_text[i] for i in idx]
else:
    p_sub, c_sub, h_sub = phate_emb, color_vals, hover_text

df_plot = pd.DataFrame({
    'PHATE-1': p_sub[:, 0],
    'PHATE-2': p_sub[:, 1],
    color_label: c_sub.astype(str),
    'title': h_sub,
})

fig = px.scatter(
    df_plot, x='PHATE-1', y='PHATE-2',
    color=color_label,
    hover_name='title',
    title='PHATE — 120k article embeddings',
    width=950, height=700,
    opacity=0.6,
    template='plotly_white',
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(legend=dict(itemsizing='constant'))
fig.show()

out_html = DRIVE_OUTPUT / 'phate_2d.html'
fig.write_html(out_html)
print(f'Interactive plot saved → {out_html}')

## Cell 11 — Static PNG  *(all 120k points)*

In [ ]:
fig_s, ax = plt.subplots(figsize=(11, 9))
sc = ax.scatter(
    phate_emb[:, 0], phate_emb[:, 1],
    c=labels, cmap='tab20',
    s=0.5, alpha=0.3, linewidths=0, rasterized=True,
)
plt.colorbar(sc, ax=ax, label='cluster', pad=0.01)
ax.set_xlabel('PHATE-1');  ax.set_ylabel('PHATE-2')
ax.set_title('PHATE — 120k article embeddings (all points)')
ax.set_aspect('equal')
plt.tight_layout()

out_png = DRIVE_OUTPUT / 'phate_2d.png'
plt.savefig(out_png, dpi=200)
plt.show()
print(f'PNG saved → {out_png}')

## Cell 12 — Save results to Drive

In [13]:
np.savez_compressed(
    DRIVE_OUTPUT / 'phate_results_modified_HDBSCAN.npz',
    phate_embedding=phate_emb,
    cluster_labels=labels,
)
meta.to_csv(DRIVE_OUTPUT / 'phate_metadata_with_clusters_modified_HDBSCAN.csv', index=False)

print('All outputs saved to', DRIVE_OUTPUT)
print('  phate_2d.html                    — interactive Plotly')
print('  phate_2d.png                     — full-resolution static PNG')
print('  phate_results.npz                — PHATE coords + cluster labels')
print('  phate_metadata_with_clusters.csv — metadata with cluster column')
print('  embeddings_cache.npz             — cached embeddings (reuse on re-run)')
print('  pca_scree.png                    — variance-explained curve')

All outputs saved to /content/drive/MyDrive/phate_output
  phate_2d.html                    — interactive Plotly
  phate_2d.png                     — full-resolution static PNG
  phate_results.npz                — PHATE coords + cluster labels
  phate_metadata_with_clusters.csv — metadata with cluster column
  embeddings_cache.npz             — cached embeddings (reuse on re-run)
  pca_scree.png                    — variance-explained curve


## Cell 13 — Hybrid Semantic Search
> Online query uses precomputed offline index for speed + semantic quality.
>
> **Stage 1 (fast):** embed query → cosine sim vs cluster centroids → select `TOP_CLUSTERS` nearest cluster positions.
>
> **Stage 2 (coverage):** build candidate pool =
>   - articles whose **hard cluster** is in top clusters  
>   - **UNION** articles that list any top-cluster position in their precomputed top-k memberships (overlap robustness)
>   - optionally noise points (label == -1)
>
> **Stage 3 (quality):** cosine rerank all candidates against query embedding → return top-K.


In [38]:
# ── Cell 13 — Hybrid Semantic Search ─────────────────────────────────────────
# Online:  query → nearest cluster positions (Stage 1, O(K) cosine ops)
#          → candidate pool: hard-cluster members UNION overlap members (Stage 2)
#          → cosine rerank against query embedding (Stage 3)
#
# Prerequisites (offline precompute in Cell 9):
#   embeddings, labels, meta,
#   cluster_centroids, cluster_id_map,
#   cluster_to_article_ids, article_top_cluster_pos,
#   MODEL, DEVICE

import textwrap
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── Search config ─────────────────────────────────────────────────────────────
SEARCH_QUERY      = input('🔍 Enter search query: ').strip()
TOP_K             = 20    # final results to return
TOP_CLUSTERS      = 5     # how many nearest clusters to select in Stage 1
INCLUDE_NOISE     = True  # also include noise points (label == -1) in candidate pool
SHOW_PHATE_COORDS = False
# ─────────────────────────────────────────────────────────────────────────────

if not SEARCH_QUERY:
    print('⚠  No query entered — skipping.')
else:
    # ── Stage 1: embed query → nearest cluster positions ──────────────────────
    print('Embedding query ...')
    _search_model = SentenceTransformer(MODEL, device=DEVICE)
    query_emb = _search_model.encode(
        [SEARCH_QUERY],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)                                   # (1, dim)

    # K cosine comparisons against precomputed centroids
    centroid_sims   = cosine_similarity(query_emb, cluster_centroids)[0]  # (K,)
    top_cluster_pos = np.argsort(centroid_sims)[::-1][:TOP_CLUSTERS]      # positions
    top_cluster_ids = cluster_id_map[top_cluster_pos]                     # real labels
    top_pos_set     = set(top_cluster_pos.tolist())

    print(f'Top {TOP_CLUSTERS} clusters by centroid cosine similarity:')
    for pos, cid in zip(top_cluster_pos, top_cluster_ids):
        size = len(cluster_to_article_ids.get(int(cid), []))
        print(f'  cluster {cid:>4}  centroid_sim={centroid_sims[pos]:.4f}  size={size:,}')

    # ── Stage 2: build candidate pool via union of two sources ────────────────
    # Source A — articles whose hard-assigned cluster is in the top set
    hard_mask = np.isin(labels, top_cluster_ids)

    # Source B — articles that "overlap" into a top cluster via their
    #             precomputed top-k nearest cluster memberships.
    #   article_top_cluster_pos[i] contains TOP_K_CLUSTERS_PER_ARTICLE positions.
    #   If ANY of those positions is in top_pos_set → include article i.
    overlap_mask = np.any(
        np.isin(article_top_cluster_pos, np.array(list(top_pos_set), dtype=np.int32)),
        axis=1,
    )

    candidate_mask = hard_mask | overlap_mask

    if INCLUDE_NOISE:
        noise_mask     = labels == -1
        candidate_mask = candidate_mask | noise_mask
        print(
            f'Candidate pool: {hard_mask.sum():,} hard-cluster'
            f' + {(overlap_mask & ~hard_mask).sum():,} overlap-only'
            f' + {noise_mask.sum():,} noise'
            f' = {candidate_mask.sum():,} total'
        )
    else:
        print(
            f'Candidate pool: {hard_mask.sum():,} hard-cluster'
            f' + {(overlap_mask & ~hard_mask).sum():,} overlap-only'
            f' = {candidate_mask.sum():,} total  (noise excluded)'
        )

    candidate_emb = embeddings[candidate_mask]          # already unit-norm
    candidate_idx = np.where(candidate_mask)[0]

    # ── Stage 3: cosine rerank all candidates against query ───────────────────
    cos_sims = cosine_similarity(candidate_emb, query_emb)[:, 0]   # (M,)

    top_local  = np.argsort(cos_sims)[::-1][:TOP_K]
    top_global = candidate_idx[top_local]

    # ── Display ───────────────────────────────────────────────────────────────
    print(f'\n{"═" * 72}')
    print(f'  Query   : "{SEARCH_QUERY}"')
    print(f'  Scoring : cosine rerank  |  top {TOP_K} from {candidate_mask.sum():,} candidates')
    print(f'{"═" * 72}\n')

    results_rows = []
    for rank, (li, gi) in enumerate(zip(top_local, top_global), 1):
        row    = meta.iloc[gi]
        cscore = float(cos_sims[li])
        cl     = int(row['cluster'])
        title  = str(row.get('title_nlp', ''))
        wrapped = textwrap.fill(title, width=64, subsequent_indent='          ')

        # Tag whether this hit came from hard-cluster or overlap expansion
        source_tag = 'hard' if hard_mask[gi] else ('overlap' if overlap_mask[gi] else 'noise')

        coords = ''
        if SHOW_PHATE_COORDS and 'phate_emb' in dir():
            coords = f'  PHATE=({phate_emb[gi, 0]:.3f}, {phate_emb[gi, 1]:.3f})'

        print(f'  #{rank:>2}  [cos={cscore:.4f}  cl={cl:>4}  src={source_tag}]{coords}')
        print(f'       {wrapped}')
        print()

        results_rows.append({
            'rank':          rank,
            'cosine_score':  round(cscore, 6),
            'cluster':       cl,
            'source':        source_tag,
            'title_nlp':     title,
        })

    search_results_df = pd.DataFrame(results_rows)
    print(f'Results stored in  →  search_results_df  ({len(search_results_df)} rows)')

    cluster_dist = search_results_df['cluster'].value_counts().head(8)
    print(f'\nTop clusters in results:')
    for cl, cnt in cluster_dist.items():
        bar = '█' * cnt
        print(f'  cluster {cl:>4}: {bar} ({cnt})')

    source_dist = search_results_df['source'].value_counts()
    print(f'\nHit source breakdown:')
    for src, cnt in source_dist.items():
        print(f'  {src:<8}: {cnt}')


🔍 Enter search query: Pakistan India Kashmir
Embedding query ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Top 5 clusters by centroid cosine similarity:
  cluster 2127  centroid_sim=0.7463  size=90
  cluster 2171  centroid_sim=0.6945  size=7,344
  cluster 2086  centroid_sim=0.6820  size=17
  cluster 1062  centroid_sim=0.6794  size=10
  cluster  508  centroid_sim=0.6769  size=8
Candidate pool: 7,469 hard-cluster + 909 overlap-only + 32,910 noise = 40,759 total

════════════════════════════════════════════════════════════════════════
  Query   : "Pakistan India Kashmir"
  Scoring : cosine rerank  |  top 20 from 40,759 candidates
════════════════════════════════════════════════════════════════════════

  # 1  [cos=0.7997  cl=2171  src=hard]
       India-Pakistan working for 'acceptable ' solution of Kashmir

  # 2  [cos=0.7901  cl=2171  src=hard]
       Pakistan credited in Kashmir overture

  # 3  [cos=0.7679  cl=2171  src=hard]
       Pakistan links Kashmir to granting India MFN status

  # 4  [cos=0.7627  cl=  -1  src=overlap]
       The Kashmir flashpoint

  # 5  [cos=0.7550  cl=2171  src=

## Cell 14 — PHATE & HDBSCAN on Retrieved Articles

This cell will take the articles retrieved from the hybrid search, re-project them into a new PHATE space, cluster them using HDBSCAN, and then visualize these local clusters in a 2D scatter plot.

In [15]:
# @title
# # Ensure search_results_df is available from the previous cell execution
# if 'search_results_df' not in locals():
#     print("Error: 'search_results_df' not found. Please run Cell 13 first.")
# else:
#     print(f'[5] Re-embedding and clustering {len(search_results_df)} retrieved articles.')


#     retrieved_article_indices = top_global

#     retrieved_embeddings = embeddings[retrieved_article_indices]
#     retrieved_meta = meta.iloc[retrieved_article_indices].copy()

#     # --- 1. Apply PCA to the retrieved embeddings (recalculating for retrieved data)
#     print('    Applying PCA to retrieved articles (determining components for 95% variance)...')
#     # Create a new PCA instance to find components for 95% variance in *retrieved* embeddings
#     pca_retrieved = PCA(n_components=0.95, random_state=42) # auto-finds components for 95%
#     retrieved_reduced = pca_retrieved.fit_transform(retrieved_embeddings)

#     explained_retrieved = np.cumsum(pca_retrieved.explained_variance_ratio_)
#     print(f'    PCA for retrieved articles: {pca_retrieved.n_components_} components explain {explained_retrieved[-1]*100:.1f}% variance.')
#     print(f'    Reduced embedding shape after PCA: {retrieved_reduced.shape}')


#     # --- 2. Apply PHATE to the PCA-reduced retrieved embeddings
#     print('    Fitting PHATE on retrieved articles...')
#     retrieved_phate_op = phate.PHATE(
#         n_components=PHATE_COMPONENTS,
#         knn=PHATE_KNN,
#         decay=PHATE_DECAY,
#         t=PHATE_T,
#         n_jobs=-1,
#         random_state=42,
#         verbose=False, # Suppress verbose output for this small dataset
#     )
#     retrieved_phate_emb = retrieved_phate_op.fit_transform(retrieved_reduced)
#     print(f'    PHATE embedding shape: {retrieved_phate_emb.shape}')

#     # --- 3. Apply HDBSCAN on the new PHATE embeddings
#     print('    Clustering retrieved articles with HDBSCAN...')
#     retrieved_clusterer = hdbscan.HDBSCAN(
#         min_cluster_size=max(2, HDBSCAN_MIN_CLUSTER_SIZE // 2), # Adjust min_cluster_size for smaller dataset
#         min_samples=2,
#         cluster_selection_epsilon=0.09,
#     )
#     retrieved_labels = retrieved_clusterer.fit_predict(retrieved_phate_emb)
#     retrieved_meta['retrieved_cluster'] = retrieved_labels

#     n_retrieved_clusters = len(set(retrieved_labels)) - (1 if -1 in retrieved_labels else 0)
#     noise_retrieved_pct  = (retrieved_labels == -1).mean() * 100
#     print(f'    {n_retrieved_clusters} clusters found in retrieved articles | noise: {noise_retrieved_pct:.1f}%')

#     # --- 4. Visualize the results interactively with Plotly
#     print('\n[6] Plotting retrieved articles in new PHATE space...')

#     # Prepare data for Plotly DataFrame
#     df_plot_retrieved = pd.DataFrame({
#         'PHATE-1': retrieved_phate_emb[:, 0],
#         'PHATE-2': retrieved_phate_emb[:, 1],
#         'Cluster': retrieved_labels.astype(str), # Convert to string for discrete colors
#         'Title': retrieved_meta['title_clean'].tolist(),
#     })

#     fig = px.scatter(
#         df_plot_retrieved, x='PHATE-1', y='PHATE-2',
#         color='Cluster',
#         hover_name='Title',
#         title=f'PHATE Visualization of {len(retrieved_meta)} Retrieved Articles (Interactive)',
#         width=950, height=700,
#         opacity=0.8,
#         template='plotly_white',
#     )
#     fig.update_traces(marker=dict(size=10)) # Make points larger for better visibility in interactive plot
#     fig.update_layout(legend=dict(itemsizing='constant'))
#     fig.show()

## Cell 15 — Bias Detection Pipeline on Retrieved Results


In [ ]:
!pip install anthropic

In [39]:
# ── Media Bias Detection Pipeline v2 ─────────────────────────────────────────
#
# Changes from v1:
#   FIX 1  — NLI stance replaces cosine-anchor stance as primary signal
#             (facebook/bart-large-mnli via zero-shot-classification)
#   FIX 2  — Alias token normalisation: dots stripped so "U.S." → "us"
#             matches headline tokens; fixes the constant vader_delta=-0.037
#   FIX 3  — Positional agency fallback for verbless headlines
#             (dep-parse finds no SUBJ → use left-of / right-of token order)
#   FIX 4  — VADER window actor-lookup now uses normalised tokens (same fix
#             as alias tokens) so windowed path actually fires
#   FIX 5  — Fusion weights updated: NLI=0.50, agency=0.20, vader=0.20,
#             mfd2=0.10; anchor_gate removed (was compensating for weak
#             cosine signal, not needed with NLI)
#   FIX 6  — NLI batch inference with progress bar; graceful CPU fallback
#   KEEP   — LLM actor extraction, verb-directed agency, entity-anchored MFD2,
#             corpus mean-centering, register weight, hedge shrink
# ─────────────────────────────────────────────────────────────────────────────

import os, re, json, time, subprocess, sys
import numpy as np
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm

# ── Auto-install deps ─────────────────────────────────────────────────────────
def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

_ensure('vaderSentiment', 'vaderSentiment')
_ensure('spacy')
_ensure('transformers')
_ensure('torch')

import spacy
import torch
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline as hf_pipeline

# ── Config ────────────────────────────────────────────────────────────────────
TITLE_COLUMN       = 'title_nlp'
SOURCE_COLUMN      = 'source'
NEUTRAL_THRESHOLD  = 0.12
MAX_TITLES         = 500
WINDOW_WORDS       = 8
USE_LLM_FOR_ACTORS = True
NLI_BATCH_SIZE     = 32

# Fusion weights (FIX 5)
W_NLI    = 0.50
W_AGENCY = 0.20
W_VADER  = 0.20
W_MFD2   = 0.10

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE     = 0 if torch.cuda.is_available() else -1
DEVICE_STR = 'cuda' if DEVICE == 0 else 'cpu'
print(f'Device: {DEVICE_STR}')

# ── Load spaCy ────────────────────────────────────────────────────────────────
try:
    nlp = spacy.load('en_core_web_trf')
except OSError:
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download',
                           'en_core_web_trf', '-q'])
    nlp = spacy.load('en_core_web_trf')

vader = SentimentIntensityAnalyzer()

# ── MFD2 vocab ────────────────────────────────────────────────────────────────
MFD2 = {
    'care_virtue':     {'protect','safe','rescue','shield','defend','guard',
                        'mercy','compassion','nurture','aid','support','heal'},
    'care_vice':       {'harm','hurt','kill','attack','wound','endanger',
                        'destroy','torture','abuse','neglect','abandon'},
    'fairness_virtue': {'fair','justice','rights','equal','legitimate','honest',
                        'impartial','proportional','deserve','due'},
    'fairness_vice':   {'unfair','unjust','biased','corrupt','cheat','manipulate',
                        'violate','discriminate','exploit','illegal'},
    'loyalty_virtue':  {'ally','united','solidarity','patriot','faithful',
                        'committed','stand','together','defend','loyal'},
    'loyalty_vice':    {'betray','traitor','defect','abandon','backstab',
                        'disloyal','treacherous'},
    'authority_virtue':{'order','law','command','authority','discipline',
                        'mandate','sovereignty','lawful','control'},
    'authority_vice':  {'chaos','disobey','rebel','violate','defy',
                        'undermine','destabilise','destabilize','rogue'},
    'sanctity_virtue': {'sacred','holy','pure','noble','righteous','honourable',
                        'honorable','dignity','revere'},
    'sanctity_vice':   {'desecrate','defile','disgrace','degrade','profane',
                        'blaspheme','contaminate'},
}
MFD2_VIRTUE = set().union(*[v for k, v in MFD2.items() if 'virtue' in k])
MFD2_VICE   = set().union(*[v for k, v in MFD2.items() if 'vice'   in k])

NEGATIVE_PREDICATES = {
    'threat','threats','mistake','mistakes','ploy','problem','problems',
    'failure','failures','crisis','danger','risk','violation','violations',
    'aggression','provocation','escalation','tension','tensions',
    'vexing','troubling','alarming','concerning','hostile','rogue',
    'wrong','fault','blame','condemn','criticize','thwart','refuse',
    'reject','block','sanction','harm','hurt','attack','destroy','kill',
}
POSITIVE_PREDICATES = {
    'deal','agreement','progress','advance','success','achievement',
    'cooperation','dialogue','diplomacy','solution','resolve',
    'incentive','incentives','offer','opportunity','support','aid',
    'protect','safe','rescue','defend','ally','united',
}
HEDGE_WORDS = {
    'possible','possibly','perhaps','maybe','might','could','may',
    'reportedly','allegedly','apparent','apparently','suggest','suggests',
    'seem','seems','unclear','uncertain','consider','considering',
}

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 0 — Prepare corpus
# ─────────────────────────────────────────────────────────────────────────────
print('\n── Stage 0: Prepare corpus ──')

if 'search_results_df' in dir() and isinstance(search_results_df, pd.DataFrame) \
        and len(search_results_df) > 0:
    _source_df = search_results_df.copy().reset_index(drop=True)
    _has_global_idx = False
    print(f'  Source: search_results_df  ({len(_source_df)} articles)')
else:
    _source_df = meta.copy().reset_index(drop=True)
    _has_global_idx = True
    print(f'  Source: full meta  ({len(_source_df)} articles)')

if MAX_TITLES and len(_source_df) > MAX_TITLES:
    _source_df = _source_df.head(MAX_TITLES).reset_index(drop=True)

titles_list = _source_df[TITLE_COLUMN].fillna('').tolist()
N = len(titles_list)
print(f'  Titles to analyse: {N}')

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 1 — Actor extraction  (unchanged from v1)
# ─────────────────────────────────────────────────────────────────────────────
print('\n── Stage 1: Actor extraction ──')
bias_meta = None

if USE_LLM_FOR_ACTORS:
    try:
        from anthropic import Anthropic
        from google.colab import userdata
        _client = Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
        _sample  = '\n'.join(f'- {t}' for t in titles_list[:30])
        _resp = _client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=500,
            system='Return ONLY valid JSON. No prose, no markdown.',
            messages=[{'role': 'user', 'content': f"""These are news article titles:
{_sample}

Return JSON:
{{
  "topic_type": one of [conflict, political, scientific, economic, other],
  "topic_summary": "one sentence",
  "actors": ["Actor A", "Actor B"],
  "actor_a_aliases": ["alias1", "alias2"],
  "actor_b_aliases": ["alias1", "alias2"],
  "label_pos": "label when framing favours Actor A",
  "label_neg": "label when framing favours Actor B"
}}

For actor_a_aliases / actor_b_aliases: include country demonyms, government
names, capital cities, and common short forms.
Example for "United States": ["US", "U.S.", "America", "American",
"Washington", "White House", "Biden", "Obama"].
"""}]
        )
        bias_meta = json.loads(
            re.sub(r'```(?:json)?', '', _resp.content[0].text).strip()
        )
        print('  Actor extraction: LLM ✓')
    except Exception as e:
        print(f'  LLM call failed ({e}); falling back to NER.')

if not bias_meta:
    ent_counter: Counter = Counter()
    for title in titles_list[:80]:
        doc = nlp(title)
        for ent in doc.ents:
            if ent.label_ in {'GPE', 'ORG', 'PERSON', 'NORP'}:
                ent_counter[ent.text.strip()] += 1
    top_ents = [e for e, _ in ent_counter.most_common(20)]
    filtered = []
    for e in top_ents:
        if not any(e != f and e.lower() in f.lower() for f in top_ents):
            filtered.append(e)
        if len(filtered) == 2:
            break
    if len(filtered) < 2:
        filtered = (filtered + ['Side A', 'Side B'])[:2]
    bias_meta = {
        'topic_type': 'other',
        'topic_summary': f'{filtered[0]} vs {filtered[1]}',
        'actors': filtered,
        'actor_a_aliases': [],
        'actor_b_aliases': [],
        'label_pos': f'pro-{filtered[0]}',
        'label_neg': f'pro-{filtered[1]}',
    }
    print(f'  Actor extraction: NER heuristic ✓  ({filtered})')

ACTOR_A   = bias_meta['actors'][0]
ACTOR_B   = bias_meta['actors'][1]
LABEL_POS = bias_meta.get('label_pos', f'pro-{ACTOR_A}')
LABEL_NEG = bias_meta.get('label_neg', f'pro-{ACTOR_B}')


# ── FIX 2: Alias token normalisation — strip dots so "U.S." → "us" ──────────
def _alias_tokens(actor: str, aliases: list) -> set:
    """
    Build a set of normalised tokens for an actor and its aliases.
    Dots are stripped BEFORE the token is stored, so "U.S." and "us" both
    become "us" and match correctly against headline tokens processed the
    same way.
    """
    STOPWORDS = {'the', 'of', 'and', 'a', 'an', 'in', 'for'}
    tokens = set()
    for phrase in [actor] + (aliases or []):
        for raw_tok in re.findall(r"[a-z'\.]+", phrase.lower()):
            tok = raw_tok.replace('.', '').rstrip("'")
            if tok and tok not in STOPWORDS:
                tokens.add(tok)
    return tokens

_a_tok = _alias_tokens(ACTOR_A, bias_meta.get('actor_a_aliases', []))
_b_tok = _alias_tokens(ACTOR_B, bias_meta.get('actor_b_aliases', []))

print(f'\n  Actor A  : {ACTOR_A}  tokens={_a_tok}')
print(f'  Actor B  : {ACTOR_B}  tokens={_b_tok}')
print(f'  Topic    : {bias_meta["topic_summary"]}')


def _norm_tok(word: str) -> str:
    """Normalise a single token the same way as _alias_tokens."""
    return re.sub(r"[^a-z']", '', word.lower().replace('.', ''))


# ─────────────────────────────────────────────────────────────────────────────
# STAGE 2 — Verb-directed agency with positional fallback  (FIX 3)
# ─────────────────────────────────────────────────────────────────────────────
print('\n── Stage 2: Lexical framing ──')


def _predicate_sentiment(doc) -> float:
    tokens_lower = {t.lemma_.lower() for t in doc} | {t.text.lower() for t in doc}
    neg_hits = len(tokens_lower & NEGATIVE_PREDICATES)
    pos_hits = len(tokens_lower & POSITIVE_PREDICATES)
    for tok in doc:
        if tok.dep_ == 'neg':
            head_lemma = tok.head.lemma_.lower()
            if head_lemma in POSITIVE_PREDICATES:
                neg_hits += 2
                pos_hits  = max(0, pos_hits - 1)
            elif head_lemma in NEGATIVE_PREDICATES:
                pos_hits += 1
    if neg_hits > pos_hits: return -1.0
    if pos_hits > neg_hits: return  1.0
    return 0.0


def _dep_agency(doc, title_tokens_norm: list) -> tuple:
    """
    Returns (agency_score, passive, hedged).

    Primary path:  spaCy dependency parse (nsubj / dobj / nsubjpass …)
    Fallback (FIX 3): when dep-parse finds no grammatical subject for either
    actor (common in verbless headlines), use left-of / right-of token order
    as a weak positional signal (strength ±0.30).
    """
    hedged  = any(t.lemma_.lower() in HEDGE_WORDS for t in doc)
    passive = any(t.dep_ in {'nsubjpass', 'auxpass'} for t in doc)

    a_is_subj = a_is_obj = b_is_subj = b_is_obj = False

    for tok in doc:
        tl = _norm_tok(tok.text)
        if tok.dep_ in {'nsubj', 'nsubjpass', 'agent'}:
            if tl in _a_tok: a_is_subj = True
            if tl in _b_tok: b_is_subj = True
        if tok.dep_ in {'dobj', 'pobj', 'attr', 'oprd'}:
            if tl in _a_tok: a_is_obj = True
            if tl in _b_tok: b_is_obj = True

    for chunk in doc.noun_chunks:
        cl = chunk.text.lower()
        chunk_norm = cl.replace('.', '')
        if any(alias in chunk_norm for alias in _a_tok):
            if chunk.root.dep_ in {'nsubj', 'nsubjpass', 'agent'}: a_is_subj = True
            if chunk.root.dep_ in {'dobj', 'pobj', 'attr'}:        a_is_obj  = True
        if any(alias in chunk_norm for alias in _b_tok):
            if chunk.root.dep_ in {'nsubj', 'nsubjpass', 'agent'}: b_is_subj = True
            if chunk.root.dep_ in {'dobj', 'pobj', 'attr'}:        b_is_obj  = True

    if   a_is_subj and not b_is_subj: gram_dir =  1.0
    elif b_is_subj and not a_is_subj: gram_dir = -1.0
    elif a_is_subj and b_is_subj:     gram_dir =  0.0
    elif a_is_obj  and not b_is_obj:  gram_dir = -0.4
    elif b_is_obj  and not a_is_obj:  gram_dir =  0.4
    else:
        # ── FIX 3: positional fallback for verbless headlines ──────────────
        first_a = next((i for i, t in enumerate(title_tokens_norm)
                        if t in _a_tok), None)
        first_b = next((i for i, t in enumerate(title_tokens_norm)
                        if t in _b_tok), None)
        if first_a is not None and first_b is not None:
            gram_dir = 0.30 if first_a < first_b else -0.30
        elif first_a is not None:
            gram_dir =  0.20   # only A mentioned — mild positive framing
        elif first_b is not None:
            gram_dir = -0.20
        else:
            gram_dir =  0.0

    if passive: gram_dir *= 0.5

    pred_sent    = _predicate_sentiment(doc)
    agency_score = (gram_dir * pred_sent) if pred_sent != 0.0 \
                   else (gram_dir * 0.5)
    if passive: agency_score *= 0.8

    return round(agency_score, 4), passive, hedged


def lexical_features(title: str) -> dict:
    doc               = nlp(title)
    title_norm        = [_norm_tok(t.text) for t in doc]
    agency, pas, hed  = _dep_agency(doc, title_norm)
    tokens_lemma      = [t.lemma_.lower() for t in doc]
    pos_v = sum(1 for t in tokens_lemma if t in POSITIVE_PREDICATES)
    neg_v = sum(1 for t in tokens_lemma if t in NEGATIVE_PREDICATES)
    total = pos_v + neg_v
    vp    = 0.0 if total == 0 else (pos_v - neg_v) / total
    return {
        'agency_score':  agency,
        'verb_polarity': round(vp, 4),
        'has_hedge':     hed,
        'passive_voice': pas,
        'n_pos_verbs':   pos_v,
        'n_neg_verbs':   neg_v,
    }


lexical_rows = [lexical_features(t) for t in titles_list]
lexical_df   = pd.DataFrame(lexical_rows)
print(f'  Done. Passive={lexical_df["passive_voice"].sum()}  '
      f'Hedged={lexical_df["has_hedge"].sum()}  '
      f'Agency mean={lexical_df["agency_score"].mean():+.3f}')

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 3A — NLI zero-shot stance  (FIX 1 — replaces cosine-anchor primary)
#
# facebook/bart-large-mnli gives P(entailment) for each hypothesis.
# We score: P(pro-A framing) − P(pro-B framing), normalised to [−1, +1].
#
# Candidate labels are phrased to match the topic_type returned by the LLM,
# so the hypotheses stay grounded in the actual framing axis.
# ─────────────────────────────────────────────────────────────────────────────
print('\n── Stage 3A: NLI zero-shot stance ──')

_nli_pipe = hf_pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=DEVICE,
)

topic_type = bias_meta.get('topic_type', 'other')

if topic_type in {'conflict', 'political'}:
    _label_a = f'The headline portrays {ACTOR_A} favourably or {ACTOR_B} negatively'
    _label_n = 'The headline is neutral or factual'
    _label_b = f'The headline portrays {ACTOR_B} favourably or {ACTOR_A} negatively'
elif topic_type == 'economic':
    _label_a = f'The headline supports {ACTOR_A} economic position'
    _label_n = 'The headline presents both sides equally'
    _label_b = f'The headline supports {ACTOR_B} economic position'
else:
    _label_a = f'The framing of this headline favours {ACTOR_A}'
    _label_n = 'The framing of this headline is balanced'
    _label_b = f'The framing of this headline favours {ACTOR_B}'

_candidate_labels = [_label_a, _label_n, _label_b]
print(f'  Labels:\n    +: {_label_a}\n    0: {_label_n}\n    -: {_label_b}')

# Batch inference with progress bar
nli_raw = []
for i in tqdm(range(0, N, NLI_BATCH_SIZE), desc='  NLI batches'):
    batch   = titles_list[i: i + NLI_BATCH_SIZE]
    results = _nli_pipe(batch, candidate_labels=_candidate_labels,
                        multi_label=False)
    if isinstance(results, dict):        # single-item batch
        results = [results]
    for r in results:
        score_map = dict(zip(r['labels'], r['scores']))
        pa  = score_map.get(_label_a, 0.0)
        pb  = score_map.get(_label_b, 0.0)
        pn  = score_map.get(_label_n, 0.0)
        # Net stance: +1 = pro-A, -1 = pro-B
        # Down-weight when neutral probability is high
        net = (pa - pb) * (1.0 - 0.5 * pn)
        nli_raw.append(net)

nli_arr = np.array(nli_raw, dtype=np.float32)

# Corpus-mean centre so a balanced corpus reads as 0
nli_corpus_mean = nli_arr.mean()
nli_arr         = nli_arr - nli_corpus_mean
_nli_max        = np.abs(nli_arr).max() + 1e-8
nli_arr         = nli_arr / _nli_max      # normalise to [−1, +1]

print(f'  NLI corpus lean (pre-centering): {nli_corpus_mean:+.4f}')
print(f'  NLI centred: mean={nli_arr.mean():+.4f}  std={nli_arr.std():.4f}')

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 3B — Entity-windowed VADER  (FIX 4: use normalised tokens)
# ─────────────────────────────────────────────────────────────────────────────
print('\n── Stage 3B: Entity-windowed VADER ──')


def _windowed_sentiment(title: str, agency: float,
                        window: int = WINDOW_WORDS) -> float:
    words = title.split()
    # FIX 4: normalise tokens the same way as alias tokens so lookup succeeds
    lower = [_norm_tok(w) for w in words]

    def _window_score(actor_toks: set):
        idx = next((i for i, w in enumerate(lower) if w in actor_toks), None)
        if idx is None:
            return None
        s = max(0, idx - window)
        e = min(len(words), idx + window + 1)
        return vader.polarity_scores(' '.join(words[s:e]))['compound']

    sa = _window_score(_a_tok)
    sb = _window_score(_b_tok)

    if sa is not None and sb is not None:
        return (sa - sb) / 2.0
    elif sa is not None:
        return sa * 0.6
    elif sb is not None:
        return -sb * 0.6
    else:
        # Last resort: full-title VADER scaled by agency direction
        full_score = vader.polarity_scores(title)['compound']
        if abs(agency) > 0.3:
            return full_score * np.sign(agency) * 0.4
        return full_score * 0.2


vader_scores = [
    _windowed_sentiment(t, a)
    for t, a in zip(titles_list, lexical_df['agency_score'].values)
]
vader_arr = np.array(vader_scores, dtype=np.float32)
vader_arr = vader_arr - vader_arr.mean()

# Diagnostic: how often did we hit the windowed path vs fallback?
_win_hits = sum(
    1 for t in titles_list
    if any(_norm_tok(w) in _a_tok or _norm_tok(w) in _b_tok
           for w in t.split())
)
print(f'  Windowed path fired: {_win_hits}/{N} titles  '
      f'({100*_win_hits/N:.1f}%)')
print(f'  VADER centred: mean={vader_arr.mean():+.4f}  '
      f'std={vader_arr.std():.4f}')

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 3C — Entity-anchored MFD2  (unchanged from v1)
# ─────────────────────────────────────────────────────────────────────────────
print('\n── Stage 3C: Entity-anchored MFD2 ──')


def mfd2_entity_score(title: str) -> float:
    doc    = nlp(title)
    tokens = list(doc)

    a_indices = [i for i, t in enumerate(tokens)
                 if _norm_tok(t.text) in _a_tok]
    b_indices = [i for i, t in enumerate(tokens)
                 if _norm_tok(t.text) in _b_tok]

    if not a_indices and not b_indices:
        return 0.0

    moral_a = moral_b = 0.0

    for i, tok in enumerate(tokens):
        lemma = tok.lemma_.lower()
        if   lemma in MFD2_VIRTUE: valence =  1.0
        elif lemma in MFD2_VICE:   valence = -1.0
        else:                      continue

        dist_a = min((abs(i - j) for j in a_indices), default=999)
        dist_b = min((abs(i - j) for j in b_indices), default=999)

        if   dist_a < dist_b: moral_a += valence
        elif dist_b < dist_a: moral_b += valence

    total = abs(moral_a) + abs(moral_b)
    if total == 0:
        return 0.0
    return round((moral_a - moral_b) / total, 5)


mfd2_arr = np.array([mfd2_entity_score(t) for t in titles_list],
                    dtype=np.float32)
mfd2_arr = mfd2_arr - mfd2_arr.mean()
print(f'  MFD2 centred: mean={mfd2_arr.mean():+.4f}  '
      f'std={mfd2_arr.std():.4f}')

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 4 — Fusion  (FIX 5: NLI primary, no anchor_gate)
# ─────────────────────────────────────────────────────────────────────────────
print('\n── Stage 4: Fusion ──')

_ANALYTICAL_RE = re.compile(
    r'\b('
    r'what\s+(the|is|are|does|happened|caused)'
    r'|how\s+(the|to|did|does|a|an)\b'
    r'|why\s+(the|did|does|is|are)\b'
    r'|a\s+brief\s+history|history\s+of|the\s+story\s+of'
    r'|explained?|explainer|overview|background|profile\s+of'
    r'|back\s+on\s+track|what\s+.*\s+is\s+about|the\s+controversy'
    r'|isolation\s+or|engagement\s+or|\bor\s+\w+\??$|\?\s*$'
    r')',
    re.IGNORECASE
)
_STANCE_RE = re.compile(
    r'\b(ploy|threat|aggression|provocation|escalat|falls\s+short'
    r'|thwart|clash|mistake|vexing|wrong|guilty|condemn|criticiz'
    r'|reject|deny|bloc?k|sanction)\b',
    re.IGNORECASE
)


def _register_weight(title: str) -> float:
    has_analytical = bool(_ANALYTICAL_RE.search(title))
    has_stance     = bool(_STANCE_RE.search(title))
    if has_analytical and not has_stance: return 0.35
    if has_analytical and has_stance:     return 0.65
    return 1.0


register_weights = np.array([_register_weight(t) for t in titles_list])
agency           = lexical_df['agency_score'].values
hedged           = lexical_df['has_hedge'].values.astype(float)

# Weighted sum — weights scale by register but keep fixed relative proportions
w_nli    = W_NLI    * register_weights
w_agency = W_AGENCY * np.ones(N)
w_vader  = W_VADER  * np.ones(N)
w_mfd2   = W_MFD2   * np.ones(N)
w_total  = w_nli + w_agency + w_vader + w_mfd2

bias_raw = (
    w_nli    * nli_arr   +
    w_agency * agency    +
    w_vader  * vader_arr +
    w_mfd2   * mfd2_arr
) / w_total

hedge_shrink = 1.0 - 0.20 * hedged
bias_scores  = bias_raw * hedge_shrink

# Final corpus mean-centering
final_mean  = bias_scores.mean()
bias_scores = np.clip(bias_scores - final_mean, -1.0, 1.0)

print(f'  Pre-centering corpus lean: {final_mean:+.4f}')
print(f'  Bias score: mean={bias_scores.mean():+.4f}  '
      f'std={bias_scores.std():.4f}  '
      f'min={bias_scores.min():+.4f}  max={bias_scores.max():+.4f}')

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 5 — Grouping + output
# ─────────────────────────────────────────────────────────────────────────────
print('\n── Stage 5: Grouping + output ──')


def _group(score: float) -> str:
    if score >  NEUTRAL_THRESHOLD: return LABEL_POS
    if score < -NEUTRAL_THRESHOLD: return LABEL_NEG
    return 'neutral'


bias_groups = np.array([_group(s) for s in bias_scores])

bias_df = _source_df[[TITLE_COLUMN]].copy().reset_index(drop=True)
bias_df['bias_score']       = np.round(bias_scores, 5)
bias_df['group']            = bias_groups
bias_df['nli_stance']       = np.round(nli_arr, 5)
bias_df['agency_score']     = np.round(agency, 5)
bias_df['vader_delta']      = np.round(vader_arr, 5)
bias_df['mfd2_score']       = np.round(mfd2_arr, 5)
bias_df['register_weight']  = np.round(register_weights, 2)
bias_df['has_hedge']        = lexical_df['has_hedge'].values
bias_df['passive_voice']    = lexical_df['passive_voice'].values

for col in ('cluster', 'rank', SOURCE_COLUMN):
    if col and col in _source_df.columns:
        bias_df[col] = _source_df[col].values

group_counts = bias_df['group'].value_counts().to_dict()
group_means  = bias_df.groupby('group')['bias_score'].mean().to_dict()

bias_summary = {
    'actor_a':          ACTOR_A,
    'actor_b':          ACTOR_B,
    'label_positive':   LABEL_POS,
    'label_negative':   LABEL_NEG,
    'topic_type':       bias_meta['topic_type'],
    'topic_summary':    bias_meta['topic_summary'],
    'corpus_lean_raw':  round(float(final_mean), 4),
    'n_titles':         N,
    'group_counts':     group_counts,
    'group_mean_scores': group_means,
    'corpus_mean_bias': round(float(bias_scores.mean()), 4),
    'corpus_std_bias':  round(float(bias_scores.std()),  4),
    'method': 'NLI-primary + verb-agency + entity-windowed-VADER + entity-MFD2 + mean-centering',
}

if SOURCE_COLUMN and SOURCE_COLUMN in bias_df.columns:
    source_bias = (
        bias_df.groupby(SOURCE_COLUMN)['bias_score']
        .agg(['mean', 'std', 'count'])
        .rename(columns={'mean': 'mean_bias', 'std': 'std_bias', 'count': 'n'})
        .sort_values('mean_bias', ascending=False)
        .round(4)
    )
    bias_summary['source_bias'] = source_bias.to_dict(orient='index')

# ── Pretty-print summary ──────────────────────────────────────────────────────
W = 72
print(f'\n{"═"*W}')
print(f'  MEDIA BIAS ANALYSIS  —  {bias_meta["topic_summary"][:48]}')
print(f'  corpus lean (pre-centering): {final_mean:+.4f}')
print(f'{"═"*W}')
print(f'  Axis: {LABEL_POS}  ←——→  neutral  ←——→  {LABEL_NEG}')
print(f'{"─"*W}')
_max_cnt = max(group_counts.values(), default=1)
for grp in [LABEL_POS, 'neutral', LABEL_NEG]:
    cnt  = group_counts.get(grp, 0)
    mean = group_means.get(grp, 0.0)
    pct  = 100 * cnt / N if N else 0
    bar  = '█' * int(cnt * 40 / _max_cnt)
    print(f'  {grp:<22} n={cnt:>4}  ({pct:5.1f}%)  mean={mean:+.3f}  {bar}')
print(f'{"─"*W}')
print(f'  Corpus mean (post-centering): {bias_scores.mean():+.4f}')
print(f'  Corpus std                  : {bias_scores.std():.4f}')
print(f'{"═"*W}')

print('\n── Sample titles per group ──────────────────────────────────────────────')
cols = ['title_nlp', 'bias_score', 'nli_stance', 'agency_score',
        'vader_delta', 'mfd2_score']
for grp in [LABEL_POS, 'neutral', LABEL_NEG]:
    sub = bias_df[bias_df['group'] == grp].sort_values(
        'bias_score', ascending=(grp == LABEL_NEG)
    )
    if len(sub):
        print(f'\n── {grp} ──')
        print(sub[cols].head(10).to_string(index=False))

print(f'\nResults → bias_df ({len(bias_df)} rows)  |  bias_summary  |  bias_meta')

Device: cuda

── Stage 0: Prepare corpus ──
  Source: search_results_df  (20 articles)
  Titles to analyse: 20

── Stage 1: Actor extraction ──
  Actor extraction: LLM ✓

  Actor A  : India  tokens={'new', 'delhi', 'hindustan', 'congress', 'modi', 'indian', 'bjp', 'vajpayee', 'india'}
  Actor B  : Pakistan  tokens={"pakistan's", 'pakistan', 'pakistani', 'musharraf', 'rawalpindi', 'isi', 'islamabad', 'pak'}
  Topic    : India and Pakistan engage in diplomatic negotiations over the disputed Kashmir region, with both sides seeking a mutually acceptable resolution.

── Stage 2: Lexical framing ──
  Done. Passive=0  Hedged=2  Agency mean=-0.062

── Stage 3A: NLI zero-shot stance ──


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

  Labels:
    +: The headline portrays India favourably or Pakistan negatively
    0: The headline is neutral or factual
    -: The headline portrays Pakistan favourably or India negatively


  NLI batches:   0%|          | 0/1 [00:00<?, ?it/s]

  NLI corpus lean (pre-centering): -0.0535
  NLI centred: mean=-0.0000  std=0.3282

── Stage 3B: Entity-windowed VADER ──
  Windowed path fired: 18/20 titles  (90.0%)
  VADER centred: mean=-0.0000  std=0.1338

── Stage 3C: Entity-anchored MFD2 ──
  MFD2 centred: mean=+0.0000  std=0.2179

── Stage 4: Fusion ──
  Pre-centering corpus lean: -0.0144
  Bias score: mean=+0.0000  std=0.2392  min=-0.6117  max=+0.4157

── Stage 5: Grouping + output ──

════════════════════════════════════════════════════════════════════════
  MEDIA BIAS ANALYSIS  —  India and Pakistan engage in diplomatic negotiat
  corpus lean (pre-centering): -0.0144
════════════════════════════════════════════════════════════════════════
  Axis: pro-India  ←——→  neutral  ←——→  pro-Pakistan
────────────────────────────────────────────────────────────────────────
  pro-India              n=   5  ( 25.0%)  mean=+0.278  ████████████████████
  neutral                n=  10  ( 50.0%)  mean=+0.011  █████████████████████████████████